# GCN Corpus EDA

This notebook measures the extracted GCN corpus tables: one circular row, one evidence span row, and one photometry span row. It reads only the emitted Parquet tables and their manifest, and it is descriptive only: no value is cleaned, altered, or exported. The final cell is reserved for decisions after review of the measurements.

In [1]:
import json
import re
from collections import Counter
from pathlib import Path

import pandas as pd
import pyarrow.parquet as pq
from IPython.display import display
pd.set_option('display.max_rows', None)
pd.set_option('display.max_columns', None)
pd.set_option('display.width', None)
pd.set_option('display.max_colwidth', None)
repo_root = Path.cwd().resolve()
while not (repo_root / 'data' / 'interim' / 'gcn_corpus').is_dir():
    if repo_root.parent == repo_root:
        raise FileNotFoundError('data/interim/gcn_corpus was not found from the notebook working directory')
    repo_root = repo_root.parent
corpus_dir = repo_root / 'data' / 'interim' / 'gcn_corpus'
circulars = pq.read_table(corpus_dir / 'circulars.parquet').to_pandas()
evidence = pq.read_table(corpus_dir / 'evidence_spans.parquet').to_pandas()
photometry = pq.read_table(corpus_dir / 'photometry_spans.parquet').to_pandas()
manifest = json.loads((corpus_dir / 'manifest.json').read_text(encoding='utf-8'))
for name, frame in [('circulars', circulars), ('evidence_spans', evidence), ('photometry_spans', photometry)]:
    print(f'{name}: {frame.shape[0]} rows x {frame.shape[1]} columns')
if any(frame.empty for frame in [circulars, evidence, photometry]):
    raise RuntimeError(f'Zero-row input table under {corpus_dir}')
expected = {'circulars': (12012, 11), 'evidence_spans': (63277, 20), 'photometry_spans': (37795, 30)}
loaded = {'circulars': circulars, 'evidence_spans': evidence, 'photometry_spans': photometry}
control_rows = [[name, frame.shape, expected[name], frame.shape == expected[name]] for name, frame in loaded.items()]
control_rows += [['manifest evidence count', manifest['annotation_counts']['evidence_spans'], len(evidence), manifest['annotation_counts']['evidence_spans'] == len(evidence)], ['manifest photometry count', manifest['annotation_counts']['photometry_spans'], len(photometry), manifest['annotation_counts']['photometry_spans'] == len(photometry)]]
controls = pd.DataFrame(control_rows, columns=['control', 'observed', 'expected', 'PASS'])
display(controls)
if not controls['PASS'].all():
    raise RuntimeError('A table-shape or manifest-count control failed')
def populated(value):
    return not pd.isna(value) and not (isinstance(value, str) and value == '')
def state_counts(series):
    nulls = int(series.isna().sum())
    empty = int(series.map(lambda value: isinstance(value, str) and value == '').sum())
    return nulls, empty, len(series) - nulls - empty


circulars: 12012 rows x 11 columns
evidence_spans: 63277 rows x 20 columns
photometry_spans: 37795 rows x 30 columns


,control,observed,expected,PASS
0,circulars,"(12012, 11)","(12012, 11)",True
1,evidence_spans,"(63277, 20)","(63277, 20)",True
2,photometry_spans,"(37795, 30)","(37795, 30)",True
3,manifest evidence count,63277,63277,True
4,manifest photometry count,37795,37795,True


In [2]:
year_by_circular = circulars.set_index('circular_id')['year']
years = sorted(circulars['year'].unique())
evidence_year = evidence['circular_id'].map(year_by_circular)
photometry_year = photometry['circular_id'].map(year_by_circular)
glance = pd.DataFrame({
    'year': years,
    'circulars': [int((circulars['year'] == year).sum()) for year in years],
    'evidence_spans': [int((evidence_year == year).sum()) for year in years],
    'photometry_spans': [int((photometry_year == year).sum()) for year in years],
    'median_text_length': [circulars.loc[circulars['year'] == year, 'text_length'].median() for year in years],
    'circulars_edited': [int(circulars.loc[circulars['year'] == year, 'was_edited'].sum()) for year in years],
})
display(glance)
print(f"Corpus date range: {circulars['created_on_utc'].min()} to {circulars['created_on_utc'].max()}")
print(f"Totals: circulars={len(circulars)}, evidence_spans={len(evidence)}, photometry_spans={len(photometry)}")

,year,circulars,evidence_spans,photometry_spans,median_text_length,circulars_edited
0,2023,2308,11285,8249,1692.0,20
1,2024,3284,16478,11535,1689.0,112
2,2025,4528,24597,12910,1677.0,353
3,2026,1892,10917,5101,1626.5,168


Corpus date range: 2023-01-01T02:26:46+00:00 to 2026-07-19T15:10:41.186000+00:00
Totals: circulars=12012, evidence_spans=63277, photometry_spans=37795


In [3]:
def field_census(frame):
    rows = []
    for column in frame.columns:
        series = frame[column]
        nulls, empty, present = state_counts(series)
        values = series[series.map(populated)]
        if present == 0:
            summary = 'no populated values'
        elif pd.api.types.is_numeric_dtype(series):
            summary = f'min={values.min()}; max={values.max()}'
        elif pd.api.types.is_bool_dtype(series):
            summary = f'true={int(values.sum())}'
        else:
            summary = f'sample={str(values.iloc[0])[:100]!r}'
        rows.append([column, str(series.dtype), nulls, empty, present, int(values.nunique(dropna=True)), summary])
    return pd.DataFrame(rows, columns=['column', 'dtype', 'null', 'empty_string', 'populated', 'distinct', 'summary'])

def census_notice(census):
    zero = census.loc[census['populated'] == 0, 'column'].tolist()
    sparse = census.loc[census['populated'] < len(circulars) * 0.01, 'column'].tolist()
    return zero, sparse

circulars_census = field_census(circulars)
display(circulars_census)
zero_columns = circulars_census.loc[circulars_census['populated'] == 0, 'column'].tolist()
sparse_columns = circulars_census.loc[circulars_census['populated'] < len(circulars) * 0.01, 'column'].tolist()
print(f"Columns populated in no row: {zero_columns or 'none'}")
print(f"Columns populated in fewer than 1% of rows: {sparse_columns or 'none'}")

,column,dtype,null,empty_string,populated,distinct,summary
0,circular_id,int64,0,0,12012,12012,min=33130; max=45188
1,subject,object,0,0,12012,11965,sample='GRB 230101A: Fermi GBM Final Real-time Localization'
2,created_on_utc,object,0,0,12012,12012,sample='2023-01-01T02:26:46+00:00'
3,edited_on_utc,object,11359,0,653,653,sample='2026-04-24T17:51:01.049000+00:00'
4,canonical_text,object,0,0,12012,12012,sample='SUBJECT: GRB 230101A: Fermi GBM Final Real-time Localization\nDATE: 2023-01-01T02:26:46+00:00\nFROM: F'
5,text_length,int64,0,0,12012,3229,min=349; max=270778
6,text_sha256,object,0,0,12012,12012,sample='763cf8f04ab688f84e9e3e734bedea4d549b0a29abe1bb5fe46f58c0f5306f1b'
7,body_hash,object,0,0,12012,11996,sample='25e0c36867a5583efa9f0348865a53265e6be6c9d3fdd21a526f39d7291d1019'
8,year,int64,0,0,12012,4,min=2023; max=2026
9,submitter,object,0,0,12012,1033,sample='Fermi GBM Team at MSFC/Fermi-GBM <do_not_reply@GIOC.nsstc.nasa.gov>'


Columns populated in no row: none
Columns populated in fewer than 1% of rows: none


In [4]:
evidence_census = field_census(evidence)
display(evidence_census)
zero_columns = evidence_census.loc[evidence_census['populated'] == 0, 'column'].tolist()
sparse_columns = evidence_census.loc[evidence_census['populated'] < len(evidence) * 0.01, 'column'].tolist()
print(f"Columns populated in no row: {zero_columns or 'none'}")
print(f"Columns populated in fewer than 1% of rows: {sparse_columns or 'none'}")

,column,dtype,null,empty_string,populated,distinct,summary
0,circular_id,int64,0,0,63277,11878,min=33130; max=45188
1,span_index,int64,0,0,63277,1,min=0; max=0
2,source_circular_id,object,63277,0,0,0,no populated values
3,text_sha256,object,0,0,63277,11878,sample='763cf8f04ab688f84e9e3e734bedea4d549b0a29abe1bb5fe46f58c0f5306f1b'
4,span_start,int64,0,0,63277,3049,min=9; max=20232
5,span_end,int64,0,0,63277,3057,min=15; max=20241
6,text,object,0,0,63277,19254,sample='GRB 230101A'
7,label,object,0,0,63277,15,sample='EVENT_IDENTITY'
8,target,object,978,0,62299,5,sample='event'
9,certainty,object,0,0,63277,5,sample='confirmed'


Columns populated in no row: ['source_circular_id']
Columns populated in fewer than 1% of rows: ['source_circular_id']


In [5]:
photometry_census = field_census(photometry)
display(photometry_census)
zero_columns = photometry_census.loc[photometry_census['populated'] == 0, 'column'].tolist()
sparse_columns = photometry_census.loc[photometry_census['populated'] < len(photometry) * 0.01, 'column'].tolist()
print(f"Columns populated in no row: {zero_columns or 'none'}")
print(f"Columns populated in fewer than 1% of rows: {sparse_columns or 'none'}")

,column,dtype,null,empty_string,populated,distinct,summary
0,circular_id,int64,0,0,37795,3907,min=33131; max=45186
1,span_index,int64,0,0,37795,4,min=0; max=3
2,text_sha256,object,0,0,37795,3907,sample='bd0e49af12f9a2b6c66edef910251adf692350beb8d67739c5940d33afd7baac'
3,span_start,int64,0,0,37795,15964,min=367; max=270545
4,span_end,int64,0,0,37795,16052,min=374; max=270664
5,text,object,0,0,37795,35608,"sample=' 77 | 2023-01-01 02:17:51 | MASTER-SAAO | (13h 35m 00.22s , -19d 18m 53.8s) | C | '"
6,measurement_type,object,0,0,37795,2,sample='upper_limit'
7,target,object,0,0,37795,2,sample='counterpart'
8,certainty,object,0,0,37795,2,sample='confirmed'
9,magnitude_or_limit,object,0,0,37795,1473,sample='16.6'


Columns populated in no row: none
Columns populated in fewer than 1% of rows: none


In [6]:
def inventory(frame, column, layer):
    result = frame.groupby(column).agg(rows=(column, 'size'), circulars=('circular_id', 'nunique')).reset_index()
    result.insert(0, 'layer', layer)
    result = result.rename(columns={column: 'label_or_measurement_type'})
    result['pct_of_layer'] = (100 * result['rows'] / len(frame)).round(3)
    return result

label_inventory = inventory(evidence, 'label', 'evidence')
measurement_inventory = inventory(photometry, 'measurement_type', 'photometry')
label_measurement_inventory = pd.concat([label_inventory, measurement_inventory], ignore_index=True).sort_values('rows', ascending=False)
display(label_measurement_inventory)
print('Vocabulary source: values present in the emitted tables; the manifest records only non-zero output counts.')
print('Values present with zero rows: none under this observed-vocabulary source.')

,layer,label_or_measurement_type,rows,circulars,pct_of_layer
16,photometry,upper_limit,32401,2371,85.728
3,evidence,EVENT_IDENTITY,27820,11519,43.965
13,evidence,TRIGGER_INSTRUMENT,6974,5834,11.021
4,evidence,HIGH_ENERGY_PROPERTY,5973,1825,9.439
15,photometry,detection,5394,1805,14.272
7,evidence,LOCALIZATION,4912,3068,7.763
14,evidence,TRIGGER_TIME,4025,3810,6.361
1,evidence,COUNTERPART_ASSOCIATION,3070,2319,4.852
6,evidence,LIGHTCURVE_EVOLUTION,2708,1539,4.280
0,evidence,CLASSIFICATION_INTERPRETATION,2061,2004,3.257


Vocabulary source: values present in the emitted tables; the manifest records only non-zero output counts.
Values present with zero rows: none under this observed-vocabulary source.


In [7]:
evidence_layer = evidence.assign(layer='evidence')
photometry_layer = photometry.assign(layer='photometry')
annotations = pd.concat([evidence_layer, photometry_layer], ignore_index=True, sort=False)
rule_inventory = annotations.groupby(['rule_id', 'extractor_id', 'layer'], dropna=False).agg(rows=('rule_id', 'size'), circulars=('circular_id', 'nunique')).reset_index()
rule_inventory['pct'] = (100 * rule_inventory['rows'] / rule_inventory.groupby('layer')['rows'].transform('sum')).round(3)
rule_inventory = rule_inventory.sort_values(['rows', 'rule_id'], ascending=[False, True], kind='stable')
display(rule_inventory)
bands = pd.Series({
    'above_1000': int((rule_inventory['rows'] > 1000).sum()),
    '100_to_1000': int(((rule_inventory['rows'] >= 100) & (rule_inventory['rows'] <= 1000)).sum()),
    '10_to_100': int(((rule_inventory['rows'] >= 10) & (rule_inventory['rows'] < 100)).sum()),
    'below_10': int((rule_inventory['rows'] < 10).sum()),
})
print(f"Distinct rules: {len(rule_inventory)}; null rule_id rows: {int(annotations['rule_id'].isna().sum())}")
print('Rule-size bands: ' + ', '.join(f'{name}={count}' for name, count in bands.items()))

,rule_id,extractor_id,layer,rows,circulars,pct
82,photometry_row.pipe,photometry-row-v1,photometry,30275,1721,80.103
28,event_identity.grb,event-identity-v1,evidence,16566,7937,26.180
25,event_identity.ep,event-identity-v1,evidence,4225,1867,6.677
32,event_identity.sname,event-identity-v1,evidence,3532,1184,5.582
83,photometry_row.whitespace,photometry-row-v1,photometry,2939,497,7.776
53,localization.error_radius,localization-v1,evidence,2660,2370,4.204
95,trigger_instrument.fermi_gbm,trigger-instrument-v1,evidence,2266,2266,3.581
48,lightcurve_evolution.fade_decline,lightcurve-evolution-v1,evidence,1868,1080,2.952
54,localization.radec_decimal,localization-v1,evidence,1768,1594,2.794
78,photometry_prose.prose_limit_upto,photometry-prose-v1,photometry,1530,1319,4.048


Distinct rules: 113; null rule_id rows: 0
Rule-size bands: above_1000=20, 100_to_1000=43, 10_to_100=28, below_10=22


In [8]:
rare_rule_ids = rule_inventory.loc[rule_inventory['rows'] <= 2, 'rule_id'].tolist()
rare = annotations[annotations['rule_id'].isin(rare_rule_ids)].copy()
rare = rare.sort_values(['rule_id', 'circular_id', 'span_start', 'span_end', 'span_index'], kind='stable')
def populated_fields(row):
    return json.dumps({key: value for key, value in row.items() if key != 'layer' and populated(value)}, default=str, ensure_ascii=False, sort_keys=True)

rare_output = rare.assign(offsets=rare['span_start'].astype(str) + ':' + rare['span_end'].astype(str), populated_fields=rare.apply(populated_fields, axis=1))
display(rare_output[['layer', 'rule_id', 'circular_id', 'offsets', 'text', 'populated_fields']])

,layer,rule_id,circular_id,offsets,text,populated_fields
20148,evidence,classification_interpretation.extended_emission,37220,2126:2158,short GRB with extended emission,"{""certainty"": ""tentative"", ""circular_id"": 37220, ""confidence"": 1.0, ""extractor_id"": ""classification-interpretation-v1"", ""extractor_version"": ""0.1"", ""label"": ""CLASSIFICATION_INTERPRETATION"", ""method"": ""regex"", ""needs_review"": false, ""rule_id"": ""classification_interpretation.extended_emission"", ""schema_version"": ""0.1"", ""span_end"": 2158, ""span_index"": 0, ""span_start"": 2126, ""target"": ""event"", ""text"": ""short GRB with extended emission"", ""text_sha256"": ""246a921769f156d9b4b4c0451f353608dbe88a693fa999a6c976da0c475b2634"", ""value"": ""short GRB""}"
6068,evidence,classification_interpretation.inferred_phenomenon,34385,1655:1664,supernova,"{""certainty"": ""tentative"", ""circular_id"": 34385, ""confidence"": 1.0, ""extractor_id"": ""classification-interpretation-v1"", ""extractor_version"": ""0.1"", ""label"": ""CLASSIFICATION_INTERPRETATION"", ""method"": ""regex"", ""needs_review"": false, ""rule_id"": ""classification_interpretation.inferred_phenomenon"", ""schema_version"": ""0.1"", ""span_end"": 1664, ""span_index"": 0, ""span_start"": 1655, ""target"": ""event"", ""text"": ""supernova"", ""text_sha256"": ""d3271263804b368b22bec199ecf1419d7f65c1d956322b131f626ac59847b024"", ""value"": ""supernova""}"
56935,evidence,classification_interpretation.inferred_phenomenon,44105,1591:1600,supernova,"{""certainty"": ""tentative"", ""circular_id"": 44105, ""confidence"": 1.0, ""extractor_id"": ""classification-interpretation-v1"", ""extractor_version"": ""0.1"", ""label"": ""CLASSIFICATION_INTERPRETATION"", ""method"": ""regex"", ""needs_review"": false, ""rule_id"": ""classification_interpretation.inferred_phenomenon"", ""schema_version"": ""0.1"", ""span_end"": 1600, ""span_index"": 0, ""span_start"": 1591, ""target"": ""event"", ""text"": ""supernova"", ""text_sha256"": ""eae2397c1a1e616ec7ae45cdafcd7aad6f3ce98a5612cd0a4ad18bb82158d439"", ""value"": ""supernova""}"
51553,evidence,classification_interpretation.this_is,43158,991:999,magnetar,"{""certainty"": ""tentative"", ""circular_id"": 43158, ""confidence"": 1.0, ""extractor_id"": ""classification-interpretation-v1"", ""extractor_version"": ""0.1"", ""label"": ""CLASSIFICATION_INTERPRETATION"", ""method"": ""regex"", ""needs_review"": false, ""rule_id"": ""classification_interpretation.this_is"", ""schema_version"": ""0.1"", ""span_end"": 999, ""span_index"": 0, ""span_start"": 991, ""target"": ""event"", ""text"": ""magnetar"", ""text_sha256"": ""acc7705d04a8f8a465ce272df15b8886e6851c1d3a98c02f7bcdbab81103f989"", ""value"": ""magnetar""}"
61527,evidence,counterpart_association.suggest,44903,2155:2203,strongly suggest this is the optical counterpart,"{""certainty"": ""candidate"", ""circular_id"": 44903, ""confidence"": 1.0, ""extractor_id"": ""counterpart-association-v1"", ""extractor_version"": ""0.1"", ""label"": ""COUNTERPART_ASSOCIATION"", ""method"": ""regex"", ""needs_review"": false, ""rule_id"": ""counterpart_association.suggest"", ""schema_version"": ""0.1"", ""span_end"": 2203, ""span_index"": 0, ""span_start"": 2155, ""target"": ""counterpart"", ""text"": ""strongly suggest this is the optical counterpart"", ""text_sha256"": ""d67473b72021580a3d185aa5477851edbcd762db03a3609b984c76ed1933651b"", ""value"": ""optical counterpart""}"
2263,evidence,duration.t50_explicit,33579,702:719,T50=9.17+/-0.04 s,"{""certainty"": ""confirmed"", ""circular_id"": 33579, ""comment"": ""T50, Konus-Wind, 100-1700 keV"", ""confidence"": 1.0, ""extractor_id"": ""duration-v1"", ""extractor_version"": ""0.1"", ""label"": ""DURATION_GENERAL"", ""method"": ""regex"", ""needs_review"": false, ""rule_id"": ""duration.t50_explicit"", ""schema_version"": ""0.1"", ""span_end"": 719, ""span_index"": 0, ""span_start"": 702, ""target"": ""event"", ""text"": ""T50=9.17+/-0.04 s"", ""text_sha25

In [9]:
circular_ids = set(circulars['circular_id'])
evidence_ids = set(evidence['circular_id'])
photometry_ids = set(photometry['circular_id'])
zero_both = circulars[circulars['circular_id'].isin(circular_ids - evidence_ids - photometry_ids)].copy()
evidence_only = circulars[circulars['circular_id'].isin((circular_ids - photometry_ids) & evidence_ids)].copy()
photometry_only = circulars[circulars['circular_id'].isin((circular_ids - evidence_ids) & photometry_ids)].copy()
coverage = pd.concat([
    zero_both.groupby('year').size().rename('circulars').reset_index().assign(group='zero_both'),
    evidence_only.groupby('year').size().rename('circulars').reset_index().assign(group='evidence_only'),
    photometry_only.groupby('year').size().rename('circulars').reset_index().assign(group='photometry_only'),
], ignore_index=True)[['group', 'year', 'circulars']].sort_values(['group', 'year'])
display(coverage)
length_report = pd.DataFrame({'group': ['zero_both', 'all_circulars'], 'min': [zero_both['text_length'].min(), circulars['text_length'].min()], 'median': [zero_both['text_length'].median(), circulars['text_length'].median()], 'max': [zero_both['text_length'].max(), circulars['text_length'].max()]})
print('Text-length distribution:\n' + length_report.to_string(index=False))
print('Zero-both subjects:\n' + zero_both[['circular_id', 'year', 'subject']].sort_values('circular_id').to_string(index=False))
terms = ['not a', 'correction', 'retraction', 'notice']
subject_counts = pd.DataFrame({'substring': terms, 'zero_both': [zero_both['subject'].str.contains(term, case=False, regex=False).sum() for term in terms], 'all_circulars': [circulars['subject'].str.contains(term, case=False, regex=False).sum() for term in terms]})
print('Subject substring counts:\n' + subject_counts.to_string(index=False))
total_by_circular = annotations.groupby('circular_id').size()
shortest = circulars.sort_values(['text_length', 'circular_id']).head(20).copy()
shortest['annotations'] = shortest['circular_id'].map(total_by_circular).fillna(0).astype(int)
print('Twenty shortest circulars:\n' + shortest[['circular_id', 'text_length', 'subject', 'annotations']].to_string(index=False))

,group,year,circulars
4,evidence_only,2023,1644
5,evidence_only,2024,2250
6,evidence_only,2025,2907
7,evidence_only,2026,1247
8,photometry_only,2023,15
9,photometry_only,2024,22
10,photometry_only,2025,28
11,photometry_only,2026,12
0,zero_both,2023,16
1,zero_both,2024,29


Text-length distribution:
        group  min  median    max
    zero_both  358   593.0   3341
all_circulars  349  1672.0 270778
Zero-both subjects:
 circular_id  year                                                                                         subject
       33299  2023 Fermi Gamma-ray Burst Monitor triggers 230206723/697396830 and 230206797/697403223 are not GRBs
       33473  2023                                                              VZLUSAT-2 detection of SGR 1806-20
       33495  2023                                                            SGR 1806-20: Correction to GCN 33494
       33638  2023          New GCN Circulars Portal for Rapid Communications on Astronomical Transients is Online
       34192  2023                                             Swift Trigger 1178410 is not an astrophysical event
       34502  2023                                                 Swift Triggers 1186280 and 1186291 are not GRBs
       34509  2023                             

In [10]:
def partial_pairs(frame):
    pairs = []
    for circular_id, group in frame.groupby('circular_id', sort=True):
        active = []
        for index, row in group.sort_values(['span_start', 'span_end', 'span_index'], kind='stable').iterrows():
            active = [(old_index, old_row) for old_index, old_row in active if old_row['span_end'] > row['span_start']]
            for old_index, old_row in active:
                if (old_row['span_start'], old_row['span_end']) != (row['span_start'], row['span_end']):
                    pairs.append((circular_id, old_index, index))
            active.append((index, row))
    return pairs
def pair_shape(first, second):
    a, b, c, d = first['span_start'], first['span_end'], second['span_start'], second['span_end']
    if (a < c and d < b) or (c < a and b < d): return 'strict_containment'
    if (a <= c and d <= b) or (c <= a and b <= d): return 'shared_boundary'
    return 'crossing'
partial_evidence = partial_pairs(evidence)
partial_photometry = partial_pairs(photometry)
exact_sizes = {name: frame.groupby(['circular_id', 'span_start', 'span_end']).size() for name, frame in [('evidence', evidence), ('photometry', photometry)]}
overlap_summary = pd.DataFrame([[name, len(partial), int((exact_sizes[name] > 1).sum()), dict(Counter(exact_sizes[name][exact_sizes[name] > 1]))] for name, partial in [('evidence', partial_evidence), ('photometry', partial_photometry)]], columns=['layer', 'partial_overlap_pairs', 'identical_offset_groups', 'identical_group_sizes'])
display(overlap_summary)
pair_rows = []
for circular_id, first_index, second_index in partial_evidence:
    first, second = evidence.loc[first_index], evidence.loc[second_index]
    outer, inner = (first, second) if first['span_start'] <= second['span_start'] and second['span_end'] <= first['span_end'] else (second, first)
    pair_rows.append([circular_id, pair_shape(first, second), outer['label'], inner['label'], first['span_start'], first['span_end'], first['label'], first['text'], second['span_start'], second['span_end'], second['label'], second['text']])
pairs = pd.DataFrame(pair_rows, columns=['circular_id', 'shape', 'outer_label', 'inner_label', 'first_start', 'first_end', 'first_label', 'first_text', 'second_start', 'second_end', 'second_label', 'second_text'])
print('Evidence partial-overlap shape counts:\n' + pairs['shape'].value_counts().to_string())
print('Evidence containment cross-tab:\n' + pairs[pairs['shape'] != 'crossing'].groupby(['outer_label', 'inner_label']).size().to_string())
print('First ten evidence partial-overlap pairs:\n' + pairs.sort_values(['circular_id', 'first_start', 'second_start']).head(10).to_string(index=False))
largest = exact_sizes['photometry'].max()
largest_keys = exact_sizes['photometry'][exact_sizes['photometry'] == largest].index.tolist()
largest_groups = []
for circular_id, start, end in largest_keys:
    group = photometry[(photometry['circular_id'] == circular_id) & (photometry['span_start'] == start) & (photometry['span_end'] == end)].copy()
    differing = [column for column in photometry.columns if group[column].nunique(dropna=False) > 1]
    group.insert(0, 'differing_fields', ', '.join(differing))
    largest_groups.append(group)
display(pd.concat(largest_groups, ignore_index=True))

,layer,partial_overlap_pairs,identical_offset_groups,identical_group_sizes
0,evidence,1251,0,{}
1,photometry,0,407,"{2: 330, 3: 67, 4: 10}"


Evidence partial-overlap shape counts:
shape
crossing              1017
shared_boundary        221
strict_containment      13
Evidence containment cross-tab:
outer_label                    inner_label                  
CLASSIFICATION_INTERPRETATION  EVENT_IDENTITY                     3
EVENT_IDENTITY                 CLASSIFICATION_INTERPRETATION     17
                               TRIGGER_INSTRUMENT               178
HIGH_ENERGY_PROPERTY           EVENT_IDENTITY                     1
                               T90                                2
LOCALIZATION                   TRIGGER_INSTRUMENT                 1
NEGATIVE_STATEMENT             EVENT_IDENTITY                    29
SPECTROSCOPY                   EVENT_IDENTITY                     2
                               HOST_CONTEXT                       1
First ten evidence partial-overlap pairs:
 circular_id    shape    outer_label                   inner_label  first_start  first_end                   first_label       

,differing_fields,circular_id,span_index,text_sha256,span_start,span_end,text,measurement_type,target,certainty,magnitude_or_limit,magnitude_error,limit_sigma,unit,photometric_band,photometric_system,obs_time_raw,obs_time_type,obs_time_reference,exposure_time_raw,instrument,instrument_provenance,comment,provenance_inherited,extractor_id,extractor_version,method,rule_id,confidence,needs_review,schema_version
0,"span_index, magnitude_or_limit",39462,0,13793ef96d58a146fa7791f7289774c398d275c6da1a49c8a34c38dcfd5c8fdf,2159,2301,| 2014974 | AT2025cpl | 86.327994 | -47.827215 | 2025-02-24 03:45:38.271 | 22.487 | 0.043 | 22.151 | 0.031 | 21.744 | 0.053 | 21.894 | 0.106 |,detection,counterpart,confirmed,22.487,0.106,None,mag,None,unknown,2025-02-24 03:45:38.271,utc_datetime,absolute_time,None,None,None,Photometry from a multi-object catalog table; verify association with the event.; missing photometric band; photometric system is unknown,[],photometry-row-v1,0.1,table-parse,photometry_row.pipe,0.7,True,0.1
1,"span_index, magnitude_or_limit",39462,1,13793ef96d58a146fa7791f7289774c398d275c6da1a49c8a34c38dcfd5c8fdf,2159,2301,| 2014974 | AT2025cpl | 86.327994 | -47.827215 | 2025-02-24 03:45:38.271 | 22.487 | 0.043 | 22.151 | 0.031 | 21.744 | 0.053 | 21.894 | 0.106 |,detection,counterpart,confirmed,22.151,0.106,None,mag,None,unknown,2025-02-24 03:45:38.271,utc_datetime,absolute_time,None,None,None,Photometry from a multi-object catalog table; verify association with the event.; missing photometric band; photometric system is unknown,[],photometry-row-v1,0.1,table-parse,photometry_row.pipe,0.7,True,0.1
2,"span_index, magnitude_or_limit",39462,2,13793ef96d58a146fa7791f7289774c398d275c6da1a49c8a34c38dcfd5c8fdf,2159,2301,| 2014974 | AT2025cpl | 86.327994 | -47.827215 | 2025-02-24 03:45:38.271 | 22.487 | 0.043 | 22.151 | 0.031 | 21.744 | 0.053 | 21.894 | 0.106 |,detection,counterpart,confirmed,21.744,0.106,None,mag,None,unknown,2025-02-24 03:45:38.271,utc_datetime,absolute_time,None,None,None,Photometry from a multi-object catalog table; verify association with the event.; missing photometric band; photometric system is unknown,[],photometry-row-v1,0.1,table-parse,photometry_row.pipe,0.7,True,0.1
3,"span_index, magnitude_or_limit",39462,3,13793ef96d58a146fa7791f7289774c398d275c6da1a49c8a34c38dcfd5c8fdf,2159,2301,| 2014974 | AT2025cpl | 86.327994 | -47.827215 | 2025-02-24 03:45:38.271 | 22.487 | 0.043 | 22.151 | 0.031 | 21.744 | 0.053 | 21.894 | 0.106 |,detection,counterpart,confirmed,21.894,0.106,None,mag,None,unknown,2025-02-24 03:45:38.271,utc_datetime,absolute_time,None,None,None,Photometry from a multi-object catalog table; verify association with the event.; missing photometric band; photometric system is unknown,[],photometry-row-v1,0.1,table-parse,photometry_row.pipe,0.7,True,0.1
4,"span_index, magnitude_or_limit",39462,0,13793ef96d58a146fa7791f7289774c398d275c6da1a49c8a34c38dcfd5c8fdf,2302,2444,| 2014991 | AT2025cpm | 86.516403 | -47.893872 | 2025-02-24 03:45:38.271 | 23.448 | 0.107 | 22.905 | 0.060 | 22.453 | 0.103 | 22.308 | 0.153 |,detection,counterpart,confirmed,23.448,0.153,None,mag,None,unknown,2025-02-24 03:45:38.271,utc_datetime,absolute_time,None,None,None,Photometry from a multi-object catalog table; verify association with the event.; missing photometric band; photometric system is unknown,[],photometry-row-v1,0.1,table-parse,photometry_row.pipe,0.7,True,0.1
5,"span_index, magnitude_or_limit",39462,1,13793ef96d58a146fa7791f7289774c398d275c6da1a49c8a34c38dcfd5c8fdf,2302,2444,| 2014991 | AT2025cpm | 86.516403 | -47.893872 | 2025-02-24 03:45:38.271 | 23.448 | 0.107 | 22.905 | 0.060 | 22.453 | 0.103 | 22.308 | 0.153 |,detection,counterpart,confirmed,22.905,0.153,None,mag,None,unknown,2025-02-24 03:45:38.271,utc_datetime,absolute_time,None,None,None,Photometry from a multi-object catalog table; verify association with the event.; missing photometric band; photometric system is unknown,[],photometry-row-v1,0.1,table-parse,photometry

In [11]:
confidence_review = annotations.groupby(['layer', 'confidence', 'needs_review'], dropna=False).size().reset_index(name='rows').sort_values(['layer', 'confidence', 'needs_review'])
display(confidence_review)
rule_review = annotations.groupby(['layer', 'rule_id'], dropna=False).agg(total_rows=('rule_id', 'size'), needs_review_rows=('needs_review', 'sum')).reset_index()
rule_review['needs_review_pct'] = (100 * rule_review['needs_review_rows'] / rule_review['total_rows']).round(3)
rule_review['review_behavior'] = rule_review.apply(lambda row: 'always' if row['needs_review_rows'] == row['total_rows'] else 'never' if row['needs_review_rows'] == 0 else 'sometimes', axis=1)
print('Rule-level review behavior:\n' + rule_review.sort_values(['layer', 'rule_id']).to_string(index=False))
for layer, frame in [('evidence', evidence), ('photometry', photometry)]:
    reviewed = frame[frame['needs_review']].copy()
    comments = reviewed.loc[reviewed['comment'].map(populated), 'comment'].value_counts().head(20)
    print(f'{layer} reviewed rows: {len(reviewed)}; top comment values (capped at 20):\n{comments.to_string()}')

,layer,confidence,needs_review,rows
0,evidence,0.50,True,5284
1,evidence,0.60,True,1
2,evidence,1.00,False,57694
3,evidence,1.00,True,298
4,photometry,0.65,True,2957
5,photometry,0.70,True,3275
6,photometry,0.90,False,1306
7,photometry,0.95,False,30257


Rule-level review behavior:
     layer                                           rule_id  total_rows  needs_review_rows  needs_review_pct review_behavior
  evidence     classification_interpretation.class_exclusion           3                  0             0.000           never
  evidence   classification_interpretation.extended_emission           1                  0             0.000           never
  evidence classification_interpretation.firm_classification           4                  0             0.000           never
  evidence           classification_interpretation.grb_class        1073                  0             0.000           never
  evidence classification_interpretation.inferred_phenomenon           2                  0             0.000           never
  evidence      classification_interpretation.interpretation          51                  0             0.000           never
  evidence      classification_interpretation.physical_cause          54                  

In [12]:
evidence_value_rows = []
for column in ['value', 'unit']:
    values = evidence.loc[evidence[column].map(populated), column]
    evidence_value_rows.append([column, len(values), values.nunique(), values.value_counts().head(20).to_dict()])
evidence_value_report = pd.DataFrame(evidence_value_rows, columns=['column', 'populated_rows', 'distinct_values', 'top_20_values'])
display(evidence_value_report)
numeric_rows = []
for column in ['magnitude_or_limit', 'magnitude_error', 'limit_sigma', 'exposure_time_raw']:
    values = photometry.loc[photometry[column].map(populated), column].astype(str)
    parsed = pd.to_numeric(values, errors='coerce')
    invalid = values[parsed.isna()]
    numeric_rows.append([column, len(values), int(parsed.notna().sum()), len(invalid), invalid.drop_duplicates().head(30).tolist(), invalid.nunique()])
numeric_report = pd.DataFrame(numeric_rows, columns=['column', 'populated_rows', 'parse_as_float', 'non_numeric_rows', 'non_numeric_values_capped_30', 'distinct_non_numeric_values'])
print('Photometry numeric parsing:\n' + numeric_report.to_string(index=False))

,column,populated_rows,distinct_values,top_20_values
0,value,60957,16206,"{'Fermi/GBM': 2266, 'fading': 1868, 'optical counterpart': 1663, 'Swift/BAT': 1388, 'long GRB': 1374, 'EP/WXT': 775, 'SVOM/ECLAIRs': 646, 'afterglow': 494, 'short GRB': 490, 'rebrightening': 412, 'Konus-Wind': 373, 'SVOM/GRM': 367, 'optical afterglow': 344, '3': 326, 'counterpart': 326, 'variable': 258, 'S240422ed': 218, 'IceCube': 217, 'GECAM': 206, '10': 200}"
1,unit,8542,16,"{'s': 1782, 'deg': 1636, 'erg/cm^2': 996, 'keV': 939, 'arcsec': 921, 'arcmin': 684, 'sec': 538, 'ph/s/cm^2': 482, 'erg/cm^2/s': 328, 'seconds': 113, 'erg': 89, 'ms': 12, 'mjd': 9, 'minutes': 9, 'MeV': 3, 'min': 1}"


Photometry numeric parsing:
            column  populated_rows  parse_as_float  non_numeric_rows                                                                                                                                                                                                                                                                                                                                                 non_numeric_values_capped_30  distinct_non_numeric_values
magnitude_or_limit           37795           37795                 0                                                                                                                                                                                                                                                                                                                                                                           []                            0
   magnitude_error            3976            

In [13]:
provenance_type = str(pq.read_schema(corpus_dir / 'photometry_spans.parquet').field('provenance_inherited').type)
raw_provenance = photometry['provenance_inherited']
non_null_provenance = raw_provenance[raw_provenance.notna()]
parsed_provenance = non_null_provenance.map(json.loads)
if not parsed_provenance.map(lambda value: isinstance(value, list)).all():
    raise TypeError('provenance_inherited contains a non-list JSON value')
lengths = parsed_provenance.map(len)
provenance_report = pd.DataFrame({
    'storage_type': [provenance_type],
    'empty_lists': [int((lengths == 0).sum())],
    'non_empty_lists': [int((lengths > 0).sum())],
    'nulls': [int(raw_provenance.isna().sum())],
    'distinct_serialized_values': [int(non_null_provenance.nunique())],
    'list_length_distribution': [lengths.value_counts().sort_index().to_dict()],
    'top_20_values': [non_null_provenance[lengths > 0].value_counts().head(20).to_dict()],
})
display(provenance_report)

,storage_type,empty_lists,non_empty_lists,nulls,distinct_serialized_values,list_length_distribution,top_20_values
0,string,6882,30913,0,20399,"{0: 6882, 1: 3075, 2: 27697, 3: 136, 4: 5}","{'[""system_expected_unknown_for_clear_unfiltered""]': 1590, '[""photometric_system=context:AB""]': 168, '[""combined_time_cols=0,1""]': 65, '[""system_from_uvot_convention""]': 36, '[""secondary_time_col=0:300(relative_to_trigger)""]': 30, '[""secondary_time_col=6:60994.17(mjd)""]': 24, '[""photometric_system=context:Vega""]': 23, '[""secondary_time_col=6:60994.14(mjd)""]': 22, '[""secondary_time_col=6:60994.18(mjd)""]': 20, '[""secondary_time_col=6:60994.16(mjd)""]': 18, '[""secondary_time_col=2:300(relative_to_trigger)""]': 18, '[""secondary_time_col=6:60994.15(mjd)""]': 18, '[""secondary_time_col=0:209(relative_to_trigger)"",""system_expected_unknown_for_clear_unfiltered""]': 14, '[""secondary_time_col=0:201(relative_to_trigger)"",""system_expected_unknown_for_clear_unfiltered""]': 12, '[""secondary_time_col=0:159(relative_to_trigger)"",""system_expected_unknown_for_clear_unfiltered""]': 12, '[""secondary_time_col=0:723(relative_to_trigger)"",""system_expected_unknown_for_clear_unfiltered""]': 11, '[""secondary_time_col=0:205(relative_to_trigger)"",""system_expected_unknown_for_clear_unfiltered""]': 11, '[""secondary_time_col=0:150(relative_to_trigger)"",""system_expected_unknown_for_clear_unfiltered""]': 11, '[""secondary_time_col=0:92(relative_to_trigger)"",""system_expected_unknown_for_clear_unfiltered""]': 10, '[""secondary_time_col=0:296(relative_to_trigger)"",""system_expected_unknown_for_clear_unfiltered""]': 10}"


In [14]:
canonical = circulars.set_index('circular_id')[['canonical_text', 'text_sha256', 'text_length']]
verification_rows, failures = [], []
for layer, frame in [('evidence', evidence), ('photometry', photometry)]:
    joined = frame.join(canonical, on='circular_id', rsuffix='_canonical')
    selected = [text[start:end] for text, start, end in zip(joined['canonical_text'], joined['span_start'], joined['span_end'])]
    text_failures = pd.Series(selected, index=joined.index) != joined['text']
    hash_failures = joined['text_sha256'] != joined['text_sha256_canonical']
    invalid_spans = joined['span_start'] >= joined['span_end']
    outside = (joined['span_start'] < 0) | (joined['span_end'] > joined['text_length'])
    failed = joined[text_failures | hash_failures | invalid_spans | outside].copy()
    if not failed.empty:
        failures.append(failed.assign(layer=layer))
    verification_rows.append([layer, len(joined), int(text_failures.sum()), int(hash_failures.sum()), int(invalid_spans.sum()), int(outside.sum())])
verification = pd.DataFrame(verification_rows, columns=['layer', 'annotations_verified', 'text_failures', 'sha256_failures', 'span_start_ge_end', 'outside_text_length'])
display(verification)
if failures:
    failed_output = pd.concat(failures)[['layer', 'circular_id', 'span_start', 'span_end', 'text']]
    print('Verification failures:\n' + failed_output.to_string(index=False))
else:
    print('Verification failures: none')

,layer,annotations_verified,text_failures,sha256_failures,span_start_ge_end,outside_text_length
0,evidence,63277,0,0,0,0
1,photometry,37795,0,0,0,0


Verification failures: none


In [15]:
created = pd.to_datetime(circulars['created_on_utc'], utc=True, format='mixed')
edited = pd.to_datetime(circulars['edited_on_utc'], utc=True, format='mixed')
gap_hours = (edited - created).dt.total_seconds() / 3600
edited_ids = set(circulars.loc[circulars['was_edited'], 'circular_id'])
time_report = pd.DataFrame([
    ['created_on_utc', str(created.dtype), str(created.dt.tz), int(created.isna().sum()), created.min(), created.max(), None, None, None, None],
    ['edited_on_utc', str(edited.dtype), str(edited.dt.tz), int(edited.isna().sum()), edited.min(), edited.max(), None, None, None, None],
    ['edit_gap_hours', 'float64', 'not_applicable', int(gap_hours.isna().sum()), None, None, gap_hours.min(), gap_hours.quantile(.25), gap_hours.median(), gap_hours.max()],
    ['edited_annotation_rows', 'int64', 'not_applicable', 0, None, None, int(evidence['circular_id'].isin(edited_ids).sum() + photometry['circular_id'].isin(edited_ids).sum()), None, None, None],
], columns=['field', 'dtype', 'timezone', 'null_count', 'min', 'max', 'value_or_gap_min', 'gap_p25', 'gap_median', 'gap_max'])
display(time_report)
print(f"Edited circulars: {int(circulars['was_edited'].sum())}")

,field,dtype,timezone,null_count,min,max,value_or_gap_min,gap_p25,gap_median,gap_max
0,created_on_utc,"datetime64[ns, UTC]",UTC,0,2023-01-01 02:26:46+00:00,2026-07-19 15:10:41.186000+00:00,NaN,NaN,NaN,NaN
1,edited_on_utc,"datetime64[ns, UTC]",UTC,11359,2024-04-03 18:46:19.777000+00:00,2026-07-15 13:28:22.527000+00:00,NaN,NaN,NaN,NaN
2,edit_gap_hours,float64,not_applicable,11359,NaT,NaT,0.035096,4.089521,14.989129,28791.059736
3,edited_annotation_rows,int64,not_applicable,0,NaT,NaT,5437.000000,NaN,NaN,NaN


Edited circulars: 653


In [16]:
candidate_rows = []
frames = {'circulars': circulars, 'evidence_spans': evidence, 'photometry_spans': photometry}
for table, frame in frames.items():
    for column in frame.columns:
        values = frame[column]
        present = values[values.map(populated)]
        if len(present) == 0:
            candidate_rows.append([table, column, 'populated_in_no_row', 0, 'no populated values'])
        types = sorted({type(value).__name__ for value in present})
        if len(types) > 1:
            candidate_rows.append([table, column, 'python_type_varies', len(present), ', '.join(types)])
        if values.dtype == object:
            whitespace = present[present.map(lambda value: isinstance(value, str) and value != value.strip())]
            if len(whitespace): candidate_rows.append([table, column, 'leading_or_trailing_whitespace', len(whitespace), str(whitespace.iloc[0])[:120]])
            replacement = present[present.map(lambda value: isinstance(value, str) and '\ufffd' in value)]
            if len(replacement): candidate_rows.append([table, column, 'contains_U+FFFD', len(replacement), str(replacement.iloc[0])[:120]])
        if re.search(r'magnitude|sigma|exposure_time|span_|confidence|text_length|year', column):
            non_numeric = present[pd.to_numeric(present.astype(str), errors='coerce').isna()]
            if len(non_numeric): candidate_rows.append([table, column, 'numeric_looking_non_numeric_values', len(non_numeric), str(non_numeric.drop_duplicates().head(3).tolist())])
    if 'comment' in frame:
        comments = frame.loc[frame['comment'].map(populated), 'comment'].astype(str)
        normalized = comments.str.lower().str.replace(r'[^a-z0-9]+', '', regex=True)
        variants = pd.DataFrame({'raw': comments, 'normalized': normalized}).groupby('normalized')['raw'].agg(lambda values: sorted(set(values)))
        duplicates = variants[variants.map(len) > 1]
        if len(duplicates): candidate_rows.append([table, 'comment', 'near_duplicate_case_or_punctuation', int(sum(len(values) for values in duplicates)), f'{len(duplicates)} groups; {duplicates.iloc[0][:3]}'])
normalization_candidates = pd.DataFrame(candidate_rows, columns=['table', 'column', 'issue', 'affected_rows', 'detail']).sort_values(['table', 'column', 'issue'], kind='stable')
display(normalization_candidates)

,table,column,issue,affected_rows,detail
2,circulars,canonical_text,contains_U+FFFD,39,SUBJECT: GRB 221231A: Swift/BAT-GUANO arcminute localization of a possibly short burst\nDATE: 2023-01-01T06:08:19+00:00\nF
1,circulars,canonical_text,leading_or_trailing_whitespace,6944,SUBJECT: GRB 230415A: Swift/BAT-GUANO detection\nDATE: 2023-04-18T05:44:11.009000+00:00\nFROM: Aaron Tohuvavohu at Univers
0,circulars,subject,leading_or_trailing_whitespace,309,GRB 230520A: MAXI/GSC detection
5,evidence_spans,comment,near_duplicate_case_or_punctuation,37,"18 groups; ['0.3 - 10 keV', '0.3-10 keV']"
3,evidence_spans,source_circular_id,populated_in_no_row,0,no populated values
4,evidence_spans,text,leading_or_trailing_whitespace,176,03:34:17.508
11,photometry_spans,exposure_time_raw,leading_or_trailing_whitespace,255,8x240s
12,photometry_spans,exposure_time_raw,numeric_looking_non_numeric_values,2572,"['1 x 20', '8 x 300 (stacked)', '5 x 300 (stacked)']"
10,photometry_spans,obs_time_raw,contains_U+FFFD,1,T+7.8 hrs ������
9,photometry_spans,obs_time_raw,leading_or_trailing_whitespace,557,2023-02-05 01:19:33


## Decisions taken from these measurements

These decisions were taken by reading the tables above. Each one names
the finding that motivates it and the number of rows it affects. They
are applied by `scripts/gcn_corpus/02_normalise.py`, and their effect is
measured in notebook C.

This corpus is built with the extraction rules as they stand today.
Where a measurement reveals a defect in a rule, the corpus records the
output as produced and the defect is documented, not repaired: repairing
it here would mean the corpus no longer reflects what the rules emit.

| # | Decision | Motivating finding | Scope |
|---|---|---|---|
| 1 | An annotation is keyed by `(circular_id, layer, span_start, span_end, span_index)`. | 407 photometry offset ranges carry more than one annotation — 330 hold two, 67 hold three, 10 hold four — because a single table row reports several filters. Offsets alone are not unique: dropping `span_index` would collapse 494 annotations. | both span tables |
| 2 | Drop `source_circular_id`. | Declared on the evidence model and populated on 0 of 63,277 rows. No rule writes it. | 1 column |
| 3 | Retain every overlapping span. Add `is_overlapping`, true where a span intersects another in the same circular and layer. | 1,251 partially overlapping evidence pairs: 1,017 crossings, 221 sharing a boundary, 13 strictly nested. The commonest shape is `EVENT_IDENTITY` containing `TRIGGER_INSTRUMENT`, 178 times, as in `Fermi GBM GRB 250129A`. Photometry has none partial and 407 at identical offsets. | 1,251 pairs |
| 4 | Retain circulars that produced no annotation. | 57 circulars yield nothing in either layer. 29 of them carry `not a` in their subject against 241 across all 12,012, and they cite trigger numbers rather than event names. Length does not explain it: among the 20 shortest circulars in the corpus, some produce six annotations and others none. | 57 rows |
| 5 | Retain `needs_review` and `comment` unchanged. | 11,815 annotations are flagged for review, and every one carries a comment stating what the rule could not resolve. 298 of them sit at confidence 1.0, all `TRIGGER_TIME`, all reporting that several candidate times share the trigger context. | 11,815 rows |
| 6 | Flag rows whose own text or fields contain U+FFFD with `has_mojibake`. Values are not repaired. | 4 annotations carry the replacement character, three of them in `photometric_band` where `g'`, `r'` and `i'` were corrupted. 39 circulars carry 516 occurrences in their canonical text, from typographic apostrophes and arcminute primes. The original character is unrecoverable. | 4 rows |
| 7 | Add `exposure_time_numeric` beside `exposure_time_raw`, holding the parsed float or null. The original is retained. | 2,572 of 32,540 populated values do not parse, across 1,110 distinct forms such as `8 x 300 (stacked)` and `2*350s (co-added)`. A null in the typed column records that the value was not a number rather than that it was absent. | 32,540 rows |
| 8 | Normalise no free-text value. No mapping, no trimming, no recoding of band, instrument, value, unit or comment. | `photometric_band` holds four populations that cannot be treated as a closed band vocabulary: the same filter under two apostrophe forms — `r'` on 171 rows against `r’` on 3 — table cells left uncut that carry the next column with them, as in `m625\t17.64`, on 18 rows, two magnitudes landing in the band column, and 54 distinct values on 180 rows that are not filters at all, including `Lim`, `n/d`, `4x180s` and bare clock times such as `13:54:44`. A further 2,837 rows are null. Separating them means correcting the extractor, not cleaning the corpus. `value` holds 16,206 distinct forms and `instrument` 282. | no change |
| 9 | Retain `text` byte for byte, trailing whitespace included. | 30,685 photometry spans and 176 evidence spans carry leading or trailing whitespace because a table-row span includes its line break. Trimming would break the property that `canonical_text[span_start:span_end]` equals `text`, which every annotation in the corpus satisfies. | 30,861 rows |
| 10 | `created_on_utc` is the time a circular became public. Add `was_edited`, true where an edit timestamp differs from it. | 653 circulars carry a later edit timestamp, median 15 hours and maximum 3.3 years. The retained body is the edited one, so their text may postdate the instant that dates them. | 653 rows |
| 11 | Retain `provenance_inherited` as a serialised JSON string. | It records where a value came from when it was not in the annotation's own cell — a secondary time column, a system read from surrounding prose. Five distinct token types produce nine token-to-column checks, and the corresponding column is populated in every case, so provenance annotates a value rather than replacing one. 20,399 distinct values, because each embeds the concrete value it explains. | 37,795 rows |
| 12 | Retain the declared vocabularies whole, including values no rule emits. | Of 57 values declared across seven vocabulary fields, 17 carry no rows; the affected fields are `target`, `measurement_type`, `certainty`, `obs_time_type` and `obs_time_reference`. Four of those values were used by annotators by hand during the INCEpTION campaign even though no rule emits them. Dropping zero-row values from the schema would hide that the rules cover only part of their own vocabulary. | 17 values |
| 13 | The corpus is accompanied by a manifest naming the extractor versions, the rule inventory and the hash of each table. | 15 extractors and 113 rules produced this corpus. Two rules account for 46% of it, and six fire exactly once. Without a manifest, two generations of the corpus are indistinguishable. | 1 file |

Checks that ran and found nothing: annotations whose offsets fall
outside their circular, spans where `span_start >= span_end`, columns
whose python type varies between rows, and non-numeric values in
`magnitude_or_limit`, `magnitude_error` or `limit_sigma`.

Defects observed in the rules and recorded for correction rather than
repaired here: the table-row parser misassigns the uncertainty when a
row carries several magnitude columns, giving four annotations the same
error value; `inferred_column` provenance attributes classification
labels such as `SN_LIKE` to the instrument field on 417 rows; and no
rule fires on subjects of the form `is not an astrophysical event`,
where `negative_statement.not_grb` covers only the `not a GRB` phrasing.